# CIDA Quickstart: Latent Deliberation for Language Model Agents

**Русский** | **English** | **中文**

---

**RU.** Данный ноутбук описывает установку и использование CIDA — модульного адаптера,
встраиваемого в произвольную причинно-каузальную языковую модель (LLM) и добавляющего
слой предварительного размышления в пространстве скрытых представлений, перед тем как
модель начинает генерировать текст. CIDA не публикует архитектуру ядра — библиотека
CIDA SDK предоставляет только публичный интерфейс взаимодействия с уже обученным адаптером.

**EN.** This notebook covers installation and usage of CIDA — a modular adapter that
attaches to any causal language model and performs a round of deliberation in latent
(hidden-state) space before the model begins generating text. The internal architecture
of the CIDA core is not published; the CIDA SDK exposes only the public interface for
interacting with a pretrained adapter.

**中文。** 本笔记本介绍 CIDA 的安装与使用方法。CIDA 是一个模块化适配器,可接入任意
因果语言模型,在模型开始生成文本之前,在隐藏状态(潜在)空间中执行一轮推理。CIDA 核心
架构不对外公开;CIDA SDK 仅提供与已训练适配器交互的公共接口。

---

### Contents

1. Installation (Hugging Face only, no GitHub dependency)
2. Loading the backbone LLM and the CIDA adapter
3. Core concepts: Adaptive Latent Compute, Latent Firewall, Reliability-Weighted Tool Trust, Shared Latent Workspace
4. Building a custom agent: tools, callbacks, memory
5. Scaling and deployment guidance
6. Full example: an incident-response agent


## 1. Installation

**RU.** Все компоненты загружаются исключительно с Hugging Face Hub. Библиотека CIDA SDK
устанавливается как обычный пакет Python (`pip install cida-inference`). Закрытое ядро
CIDA (архитектура адаптера) распространяется как приватный артефакт на Hugging Face Hub
и требует токена доступа с правом чтения (`HF_TOKEN`). Веса модели и базовой LLM также
загружаются с Hugging Face.

**EN.** All components are fetched exclusively from the Hugging Face Hub. The CIDA SDK
installs as a standard Python package (`pip install cida-inference`). The proprietary
CIDA core (the adapter architecture) is distributed as a private artifact on the Hugging
Face Hub and requires a read-scoped access token (`HF_TOKEN`). Model weights and the
baseline LLM are also fetched from Hugging Face.

**中文。** 所有组件均仅从 Hugging Face Hub 获取。CIDA SDK 以标准 Python 包形式安装
(`pip install cida-inference`)。CIDA 私有核心(适配器架构)作为私有资源发布于
Hugging Face Hub,需要具有读取权限的访问令牌(`HF_TOKEN`)。模型权重与基础 LLM 同样
从 Hugging Face 获取。


In [ ]:
# -----------------------------------------------------------------------
# Configuration. Replace the placeholders below with your actual identifiers.
# -----------------------------------------------------------------------

SDK_PACKAGE_NAME   = "cida-inference"              # Public PyPI package name for the SDK
CORE_PRIVATE_REPO  = "kirtzh/cida-core-private"    # Gated Hugging Face repo hosting the private core wheel
CORE_WHEEL_FILE    = "cida_plugin-latest-py3-none-any.whl"
WEIGHTS_REPO       = "kirtzh/Kairos-7B-CIDA-v1.0"  # Hugging Face repo hosting CIDA adapter weights
BASELINE_MODEL     = "Qwen/Qwen2.5-1.5B-Instruct"  # Public baseline LLM on Hugging Face

# -----------------------------------------------------------------------
# 1. Runtime dependencies (public, PyPI only)
# -----------------------------------------------------------------------
!pip install -q torch transformers accelerate bitsandbytes huggingface_hub requests

# -----------------------------------------------------------------------
# 2. Authenticate with Hugging Face
#    A read-scoped token is required to access the private CIDA core and
#    the gated adapter weights repository.
# -----------------------------------------------------------------------
from huggingface_hub import login, hf_hub_download
import getpass

HF_TOKEN = getpass.getpass("Hugging Face access token (read scope): ").strip()
if HF_TOKEN:
    login(token=HF_TOKEN)

# -----------------------------------------------------------------------
# 3. Install the CIDA SDK (public interface library)
# -----------------------------------------------------------------------
try:
    import cida_inference  # noqa: F401
    print("CIDA SDK already available.")
except ImportError:
    !pip install -q {SDK_PACKAGE_NAME}

# -----------------------------------------------------------------------
# 4. Fetch and install the private CIDA core (proprietary architecture)
#    Distributed as a Python wheel hosted on a gated Hugging Face repo.
#    No GitHub access is required or used at any point in this notebook.
# -----------------------------------------------------------------------
try:
    import cida_plugin  # noqa: F401
    print("CIDA core already available.")
except ImportError:
    core_wheel_path = hf_hub_download(
        repo_id=CORE_PRIVATE_REPO,
        filename=CORE_WHEEL_FILE,
        token=HF_TOKEN or None,
    )
    !pip install -q "{core_wheel_path}"

import torch
print(f"PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")


## 2. Loading the Backbone LLM and the CIDA Adapter

**RU.** `CIDAWrapper.from_pretrained` загружает конфигурацию и веса адаптера напрямую
из репозитория Hugging Face, указанного в `WEIGHTS_REPO`. Никаких дополнительных файлов
с GitHub не требуется.

**EN.** `CIDAWrapper.from_pretrained` loads the adapter configuration and weights directly
from the Hugging Face repository specified in `WEIGHTS_REPO`. No additional files are
fetched from GitHub.

**中文。** `CIDAWrapper.from_pretrained` 直接从 `WEIGHTS_REPO` 指定的 Hugging Face 仓库
加载适配器配置与权重,整个过程不涉及 GitHub。


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from cida_inference.backbone import HuggingFaceBackbone
from cida_inference.loader import CIDAWrapper

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading baseline model: {BASELINE_MODEL}")
tokenizer = AutoTokenizer.from_pretrained(BASELINE_MODEL, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASELINE_MODEL, torch_dtype=torch.bfloat16, device_map="auto"
)
base_model.eval()

backbone = HuggingFaceBackbone(base_model)

print(f"Loading CIDA adapter weights from: {WEIGHTS_REPO}")
wrapper = CIDAWrapper.from_pretrained(base_model, cida_repo=WEIGHTS_REPO)

print("Backbone and CIDA adapter are ready.")


## 3. Core Concepts

**RU.** Ниже описаны возможности CIDA SDK на уровне интерфейса. Внутренняя механика
адаптера (то, как именно происходит совещание в латентном пространстве) не раскрывается —
описано только наблюдаемое поведение и то, как этим управлять через параметры SDK.

**EN.** The sections below describe CIDA SDK capabilities at the interface level. The
internal mechanics of the adapter (how the latent deliberation is actually computed) are
not disclosed — only the observable behavior and how to control it through SDK parameters.

**中文。** 以下内容在接口层面介绍 CIDA SDK 的能力。适配器内部机制(潜在空间推理的具体
计算方式)不予公开,此处仅说明可观察到的行为以及如何通过 SDK 参数进行控制。

---

### 3.1 Adaptive Latent Compute

**RU.** Число раундов внутреннего совещания не фиксировано — оно определяется динамически
на основе меры несогласия между внутренними представлениями на каждом шаге. Простые запросы
сходятся за меньшее число раундов; сложные — за большее. Управляется параметрами
`default_reasoning_rounds`, `uncertainty_threshold` при создании `LatentAgent`.

**EN.** The number of internal deliberation rounds is not fixed — it is determined
dynamically based on a disagreement measure computed at each step. Simple queries converge
in fewer rounds; harder ones take more. Controlled via the `default_reasoning_rounds` and
`uncertainty_threshold` parameters of `LatentAgent`.

**中文。** 内部推理轮数并非固定,而是根据每一步计算出的分歧程度动态决定。简单查询收敛
所需轮数较少,复杂查询则需要更多轮数。可通过 `LatentAgent` 的 `default_reasoning_rounds`
与 `uncertainty_threshold` 参数进行控制。

### 3.2 Latent Firewall

**RU.** Перед тем как модель сгенерирует хотя бы один токен действия, промежуточное
латентное представление плана сравнивается с эмбеддингами запрещённых намерений
(`guardrails`). При превышении порога сходства действие блокируется и генерация
перенаправляется, без единого потраченного токена на потенциально небезопасный вывод.

**EN.** Before the model emits a single action token, the intermediate latent plan
representation is compared against embeddings of prohibited intents (`guardrails`). If
the similarity threshold is exceeded, the action is blocked and generation is redirected —
without spending a single token on a potentially unsafe output.

**中文。** 在模型生成任何行动 token 之前,系统会将中间潜在计划表示与被禁止意图的嵌入
(`guardrails`)进行比对。若相似度超过阈值,该行动将被阻止并重新引导生成过程,不会为
潜在的不安全输出消耗任何 token。

### 3.3 Reliability-Weighted Tool Trust

**RU.** Каждый зарегистрированный инструмент имеет экспоненциально сглаженную оценку
надёжности, обновляемую по итогам каждого вызова. Инструменты, часто возвращающие ошибки,
получают меньший вес при формировании плана. Это происходит автоматически, без ручной
настройки.

**EN.** Every registered tool carries an exponentially smoothed reliability score, updated
after each call. Tools that fail frequently receive lower weight in plan formation. This
happens automatically, with no manual tuning required.

**中文。** 每个已注册的工具都拥有一个经过指数平滑处理的可靠性评分,并在每次调用后更新。
频繁失败的工具在计划生成过程中获得的权重会相应降低。此过程自动完成,无需人工调整。

### 3.4 Shared Latent Workspace

**RU.** Несколько экземпляров `LatentAgent` могут использовать общий объект `LatentMemory`.
Это позволяет им обмениваться состояниями напрямую в латентном пространстве, без обмена
текстовыми сообщениями между собой.

**EN.** Multiple `LatentAgent` instances can share a single `LatentMemory` object. This
lets them exchange state directly in latent space, without exchanging text messages with
each other.

**中文。** 多个 `LatentAgent` 实例可以共享同一个 `LatentMemory` 对象,从而直接在潜在
空间中交换状态,而无需彼此传递文本消息。


## 4. Building a Custom Agent

**RU.** `LatentAgent` принимает произвольные Python-функции в качестве инструментов.
Сигнатура функции (типы аргументов, значения по умолчанию) используется автоматически
для типизированного разбора вызовов модели — дополнительная схема не требуется.

**EN.** `LatentAgent` accepts arbitrary Python functions as tools. The function signature
(argument types, default values) is used automatically for typed parsing of model tool
calls — no additional schema is required.

**中文。** `LatentAgent` 可将任意 Python 函数作为工具。函数签名(参数类型、默认值)将
被自动用于对模型工具调用进行类型化解析,无需额外定义模式。


In [ ]:
from cida_inference.latent_agent import LatentAgent
from cida_inference.callbacks import BaseCallbackHandler

def get_current_temperature(city: str) -> str:
    \"\"\"Return the current temperature for a given city.\"\"\"
    fake_data = {"astana": "-3C", "almaty": "5C", "dubai": "34C"}
    return fake_data.get(city.lower(), "Unknown city.")

class QuickstartLogger(BaseCallbackHandler):
    def on_tool_call(self, tool_name, args, kwargs):
        print(f"Tool call: {tool_name}(args={args}, kwargs={kwargs})")

    def on_deliberation_end(self, rounds_used):
        print(f"Deliberation converged after {rounds_used} round(s).")

agent = LatentAgent(
    wrapper=wrapper,
    llm=backbone,
    tokenizer=tokenizer,
    tools={"get_current_temperature": get_current_temperature},
    callbacks=[QuickstartLogger()],
    default_reasoning_rounds=4,
)

result = agent.run("What is the current temperature in Almaty?", max_turns=4)
print("\nFinal result:", result)


## 5. Scaling and Deployment Guidance

**RU.**

- **Более крупные бэкенды.** `backbone.py` определяет протокол `LLMBackbone`. Для интеграции
  с иными средами инференса (vLLM, TGI, llama.cpp) реализуйте тот же протокол — остальной
  код (`LatentAgent`, `LatentReasoner`) не потребует изменений.
- **Персистентная память.** `LatentMemory.save_to_file` / `load_from_file` позволяют
  сохранять и восстанавливать состояние между сессиями. Для продакшн-нагрузки замените
  файловое хранилище на векторную БД по тому же интерфейсу.
- **Наблюдаемость.** Реализуйте собственный `BaseCallbackHandler` для отправки событий
  (`on_tool_call`, `on_rollback`, `on_guardrail_violation` и др.) в систему мониторинга
  (Prometheus, Datadog, любой лог-агрегатор).
- **Дообучение под задачу.** Замороженный бэкбон и небольшое число обучаемых параметров
  адаптера позволяют адаптировать CIDA к новой предметной области на ограниченном наборе
  примеров без переобучения базовой LLM. Детали процесса дообучения не входят в объём
  данного SDK.

**EN.**

- **Larger backends.** `backbone.py` defines the `LLMBackbone` protocol. To integrate with
  other inference runtimes (vLLM, TGI, llama.cpp), implement the same protocol — the rest
  of the code (`LatentAgent`, `LatentReasoner`) requires no changes.
- **Persistent memory.** `LatentMemory.save_to_file` / `load_from_file` allow saving and
  restoring state across sessions. For production workloads, replace the file-based store
  with a vector database behind the same interface.
- **Observability.** Implement a custom `BaseCallbackHandler` to forward events
  (`on_tool_call`, `on_rollback`, `on_guardrail_violation`, etc.) to your monitoring stack
  (Prometheus, Datadog, any log aggregator).
- **Domain fine-tuning.** The frozen backbone and small number of trainable adapter
  parameters allow CIDA to be adapted to a new domain on a limited set of examples without
  retraining the base LLM. The fine-tuning process itself is outside the scope of this SDK.

**中文。**

- **更大的后端。** `backbone.py` 定义了 `LLMBackbone` 协议。若需接入其他推理运行时
  (vLLM、TGI、llama.cpp),只需实现相同协议,`LatentAgent`、`LatentReasoner` 等其余代码
  无需修改。
- **持久化记忆。** `LatentMemory.save_to_file` / `load_from_file` 支持跨会话保存与恢复
  状态。生产环境中可将基于文件的存储替换为遵循相同接口的向量数据库。
- **可观测性。** 可实现自定义 `BaseCallbackHandler`,将事件(`on_tool_call`、
  `on_rollback`、`on_guardrail_violation` 等)转发至监控系统(Prometheus、Datadog 或任意
  日志聚合工具)。
- **领域微调。** 冻结的主干模型与数量较少的可训练适配器参数,使 CIDA 能够在有限样本上
  适配新领域,而无需重新训练基础 LLM。微调流程本身不在本 SDK 范围内。


## 6. Full Example: An Incident-Response Agent

**RU.** Ниже — комплексный пример, объединяющий все возможности SDK в одном сценарии:
агент диагностирует проблему на симулированном сервере, используя один надёжный и один
намеренно нестабильный инструмент (демонстрация Reliability-Weighted Tool Trust и
Latent Rollback), соблюдает ограничение (Latent Firewall) на выполнение разрушительных
команд, и — если уверенность остаётся низкой после нескольких попыток — эскалирует
задачу человеку вместо того, чтобы действовать вслепую.

**EN.** Below is a comprehensive example combining every SDK capability in a single
scenario: the agent diagnoses an issue on a simulated server using one reliable and one
intentionally flaky tool (demonstrating Reliability-Weighted Tool Trust and Latent
Rollback), respects a hard constraint (Latent Firewall) against destructive commands, and
— if uncertainty remains high after several attempts — escalates to a human rather than
acting blindly.

**中文。** 以下是一个综合示例,将 SDK 的全部能力整合于单一场景中:智能体在模拟服务器上
诊断问题,使用一个稳定工具和一个刻意设计为不稳定的工具(用以演示可靠性加权工具信任与
潜在回滚机制),同时遵守禁止执行破坏性命令的硬性约束(潜在防火墙),并在多次尝试后
仍存在较高不确定性时,将任务上报给人工处理,而非盲目执行。


In [ ]:
import random

# -----------------------------------------------------------------------
# Simulated server environment
# -----------------------------------------------------------------------
class ServerEnvironment:
    def __init__(self):
        self.metrics_call_count = 0
        self.state = {
            "cpu_load": 92,
            "memory_used_pct": 74,
            "error_rate_5xx": 0.18,
            "last_deploy_minutes_ago": 6,
        }

    def check_server_health(self, server_id: str) -> str:
        \"\"\"Return current health metrics for the given server ID.\"\"\"
        return (
            f"server={server_id} cpu_load={self.state['cpu_load']}% "
            f"memory={self.state['memory_used_pct']}% "
            f"error_rate_5xx={self.state['error_rate_5xx']} "
            f"last_deploy_minutes_ago={self.state['last_deploy_minutes_ago']}"
        )

    def query_metrics_db(self, server_id: str, window_minutes: int = 15) -> str:
        \"\"\"Query the historical metrics database for a server. Intentionally flaky.\"\"\"
        self.metrics_call_count += 1
        if self.metrics_call_count == 1:
            return "Error: metrics database connection timed out."
        return (
            f"server={server_id} window={window_minutes}m "
            f"trend=error_rate_rising deploy_correlation=high"
        )

    def restart_service(self, server_id: str, service_name: str) -> str:
        \"\"\"Restart a named service on the given server. Safe, non-destructive action.\"\"\"
        return f"SUCCESS: service '{service_name}' restarted on {server_id}."

    def rollback_last_deploy(self, server_id: str) -> str:
        \"\"\"Roll back the most recent deployment on the given server. Safe, reversible action.\"\"\"
        return f"SUCCESS: last deployment rolled back on {server_id}."

    def drop_production_database(self, server_id: str) -> str:
        \"\"\"Irreversibly drop the production database. Destructive, must never run automatically.\"\"\"
        return f"DESTRUCTIVE ACTION EXECUTED on {server_id}. This should not have happened."


env = ServerEnvironment()

# -----------------------------------------------------------------------
# Guardrails: latent firewall constraints
# -----------------------------------------------------------------------
guardrails = [
    "drop or delete the production database",
    "run destructive irreversible commands without explicit human confirmation",
    "disable monitoring or alerting systems",
]

# -----------------------------------------------------------------------
# Callbacks: structured observability for this run
# -----------------------------------------------------------------------
class IncidentLogger(BaseCallbackHandler):
    def on_deliberation_step(self, round_idx, disagreement):
        print(f"  round {round_idx + 1}: disagreement={disagreement:.4f}")

    def on_rollback(self, retry_idx, reason):
        print(f"  latent rollback #{retry_idx}: {reason}")

    def on_uncertainty_high(self, uncertainty, threshold):
        print(f"  uncertainty {uncertainty:.4f} exceeds threshold {threshold:.4f} -- "
              f"escalation path engaged")

    def on_guardrail_violation(self, similarity):
        print(f"  latent firewall blocked an action (similarity={similarity:.4f})")

    def on_tool_call(self, tool_name, args, kwargs):
        print(f"  calling tool: {tool_name}(args={args}, kwargs={kwargs})")


incident_agent = LatentAgent(
    wrapper=wrapper,
    llm=backbone,
    tokenizer=tokenizer,
    tools={
        "check_server_health": env.check_server_health,
        "query_metrics_db": env.query_metrics_db,
        "restart_service": env.restart_service,
        "rollback_last_deploy": env.rollback_last_deploy,
        "drop_production_database": env.drop_production_database,
    },
    guardrails=guardrails,
    callbacks=[IncidentLogger()],
    default_reasoning_rounds=4,
    uncertainty_threshold=0.80,
)

task = (
    "Server 'prod-eu-west-3' is showing degraded performance. Diagnose the issue using "
    "the available tools, take the least destructive corrective action that resolves it, "
    "and summarize what you did and why."
)

final_answer = incident_agent.run(task, max_turns=8)
print("\n" + "=" * 72)
print("FINAL ANSWER")
print("=" * 72)
print(final_answer)


**RU.** В этом сценарии: инструмент `query_metrics_db` намеренно возвращает ошибку
при первом вызове, что задействует Reliability-Weighted Tool Trust и Latent Rollback;
`guardrails` блокирует любую попытку вызвать `drop_production_database`, даже если
модель сочтёт это кратчайшим путём к цели; число раундов совещания варьируется в
зависимости от того, насколько однозначна ситуация на конкретном шаге.

**EN.** In this scenario: the `query_metrics_db` tool intentionally fails on its first
call, engaging Reliability-Weighted Tool Trust and Latent Rollback; `guardrails` blocks
any attempt to call `drop_production_database`, even if the model considers it the
shortest path to the goal; the number of deliberation rounds varies depending on how
unambiguous the situation is at each step.

**中文。** 在此场景中:`query_metrics_db` 工具在首次调用时被刻意设置为返回错误,以触发
可靠性加权工具信任与潜在回滚机制;`guardrails` 会阻止任何调用 `drop_production_database`
的尝试,即使模型认为这是达成目标的最短路径;推理轮数会根据每一步情境的明确程度而动态变化。
